# Notebook 08 — Stage 4: analysis audit + PASS/REVIEW/FAIL verdict

**Final stage. Analyst-facing decision layer.** Reads Stage 3's
`enhanced.csv` and produces the analysis-ready CSV plus a per-row
verdict (`analysis_audit_status` = PASS / REVIEW / FAIL) with explicit
reason codes.

```text
07_stage3.ipynb
   └── <stage3_out>/data/enhanced.csv
                            │
                            ▼
                 08_stage4.ipynb            ◄── THIS NOTEBOOK
                    • re-resolve canonical aliases (defensive)
                    • coerce + standardise types
                    • missing-key + duplicate detection
                    • range checks (long-format output)
                    • strict DIC species audit (gate, not diagnostic)
                    • PASS / REVIEW / FAIL classification with reason codes
                    • write analysis_ready.csv + all audit tables
```

**Severity ladder** (the analyst's decision logic):

- **FAIL** — missing required key, Stage 3 strict carbonate issue,
  strict DIC species-sum fail, DIC unit mismatch, unknown solver,
  unknown input pair.
- **REVIEW** — duplicate complete-key row, range flag, Stage 3
  (non-strict) carbonate issue, Stage 2 replicate conflict, DIC unit
  *missing* (vs mismatched), Stage 3 robust DIC outlier, Stage 3 pH
  diagnostic mismatch.
- **PASS** — none of the above.

This is the PASS / WARN / FAIL "quality gate" pattern (NDepend,
SonarQube, UN/ABS *Data Quality Manual Part B*). We use REVIEW for the
middle tier because that's what an analyst actually does: review the
row, decide whether to include it. **Flags are advisory** — nothing is
deleted; the analyst filters on `analysis_audit_status` downstream.

**Stage 4 does NOT** rebuild chemistry, re-filter sample rows, or
overwrite Stage 3's QC flags. It audits, classifies, and writes.


## Parameters

Single tagged `parameters` cell.

For a reusable project, the default `INPUT_CSV` and `OUT_DIR` are deliberately
set to `None`. Run this notebook through `run_pipeline.sh` or set both paths
manually before execution. Notebook 08 expects the Stage 3 handoff file:

```text
<stage3_out>/data/enhanced.csv
```


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# Stage 4 reads the Stage 3 handoff:
#     <stage3_out>/data/enhanced.csv
INPUT_CSV = None
OUT_DIR = None

# --- Config override (optional) ------------------------------------
# Deep merges onto roap.stage4.STAGE4_DEFAULTS.
CONFIG_PATH = None

# --- Stage 4 behaviour ---------------------------------------------
NO_RANGE_CHECKS = False       # Skip range checks entirely.
NO_DIC_SPECIES_CHECK = False  # Skip the strict DIC audit.
NO_PARQUET = False            # Skip Parquet writes (CSV only).
DRY_RUN = False               # Plan everything, write nothing.

# Internal PyCO2SYS carbonate calculation is OFF by default; enable it for
# real data via config (carbonate_calc.enabled: true) or set this False.
NO_CARBONATE_CALC = True


## Setup

All Stage 4 logic lives in `roap.stage4`: `run(df, stage_config)` re-resolves
aliases, coerces and standardises, optionally computes the carbonate system,
checks keys, duplicates and ranges, runs the strict DIC audit and assigns
the PASS / REVIEW / FAIL verdict, returning a `StageResult` without touching
disk. This notebook reads the input, calls `run`, shows the tables and
writes the result.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from roap import stage4
from roap.common import as_bool, die, is_none_like
from roap.stage_config import load_stage_config
from roap.stage_result import write_stage_result

## Load input and canonicalise

Defensive: re-resolve aliases (no-op when fed Stage 3's output, recovers
when fed something else), check required columns, coerce types,
re-normalise scale/unit strings.


In [ ]:
NO_RANGE_CHECKS = as_bool(NO_RANGE_CHECKS)
NO_DIC_SPECIES_CHECK = as_bool(NO_DIC_SPECIES_CHECK)
NO_PARQUET = as_bool(NO_PARQUET)
DRY_RUN = as_bool(DRY_RUN)
NO_CARBONATE_CALC = as_bool(NO_CARBONATE_CALC)

if is_none_like(INPUT_CSV):
    die("INPUT_CSV is required. Run through run_pipeline.sh or set INPUT_CSV.")
if is_none_like(OUT_DIR):
    die("OUT_DIR is required. Run through run_pipeline.sh or set OUT_DIR.")
config_path = None if is_none_like(CONFIG_PATH) else str(CONFIG_PATH).strip()

input_csv = Path(str(INPUT_CSV).strip()).expanduser().resolve()
if not input_csv.exists():
    die(f"File not found: {input_csv}\nDid Stage 3 run successfully? Stage 4 reads its enhanced.csv.")
if input_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {input_csv.name}")
out_root = Path(str(OUT_DIR).strip()).expanduser().resolve()

# Stage 4 defaults <- this Stage's optional config file, and nothing else (#41).
stage_cfg = load_stage_config("stage4", config_path)
for _warning in stage_cfg.warnings:
    print(f"WARNING: {_warning}")

# The internal PyCO2SYS calculation is OFF unless NO_CARBONATE_CALC = False
# or the config sets carbonate_calc.enabled: true (configs/08_stage4.yaml).
try:
    result = stage4.run(
        pd.read_csv(input_csv),
        stage_cfg,
        source=str(input_csv),
        range_checks=not NO_RANGE_CHECKS,
        dic_species_check=not NO_DIC_SPECIES_CHECK,
        carbonate_calc=not NO_CARBONATE_CALC,
    )
except ValueError as exc:
    die(str(exc))

result.manifest["output_root"] = str(out_root)
result.manifest["parameters"].update(
    {
        "INPUT_CSV": str(input_csv),
        "OUT_DIR": str(out_root),
        "CONFIG_PATH": CONFIG_PATH,
        "config_path_resolved": config_path,
        "NO_RANGE_CHECKS": NO_RANGE_CHECKS,
        "NO_DIC_SPECIES_CHECK": NO_DIC_SPECIES_CHECK,
        "NO_PARQUET": NO_PARQUET,
        "DRY_RUN": DRY_RUN,
    }
)
df = result.data["data/analysis_ready.csv"]
tables = result.tables
counts = result.manifest["row_counts"]

print(f"Rows loaded: {len(df):,}")
print(f"Columns    : {df.shape[1]}")
if display is not None:
    display(tables["tables/canonical_presence.csv"])

## Internal carbonate-system calculation (before the audit)

Compute the carbonate system **inside the pipeline** with PyCO2SYS, from the
RM-corrected total alkalinity (`ta_corrected_umolkg`) and measured pH, at the
lab-temperature input / in-situ-temperature output convention. This replaces
reliance on externally computed (Excel) chemistry that was based on *raw* TA,
so the reported saturation states / DIC / pCO2 are consistent with the
corrected TA. Settings are pinned and validated (Lueker 2000; Dickson KSO4;
Lee 2010 borate; Perez & Fraga KF; total pH scale). See `carbonate_calc_design.md`.

Toggle with the `NO_CARBONATE_CALC` parameter (default: run it).

In [ ]:
carbonate = result.manifest["carbonate_calc"]
print(carbonate["note"])
for _n in carbonate.get("notes", []):
    print("  -", _n)
if carbonate.get("batch_quality"):
    print("  RM batch quality:", carbonate["batch_quality"])

## Key integrity and duplicate detection

Two related but distinct checks:

- **`missing_key_rows`** — rows where any duplicate-key column is
  null or empty. These get one `flag_missing_key__<col>` per missing
  column in the audit table.
- **`detect_duplicates`** — Stage 4's stricter duplicate detector: a
  row is duplicate only if **every** key column is non-null AND the
  values collide with another row's. Rows with any null in the key
  fall through to the missing-key check, not the duplicate check.
  (This is stricter than Stage 2's flag, which is advisory and works
  on partial-key collisions.)


In [ ]:
print(f"Missing-key rows              : {counts['missing_key_rows']:,}")
print(f"Missing required-analysis rows: {counts['missing_required_analysis_rows']:,}")
print(f"Duplicate rows                : {counts['duplicate_rows']:,}")
for m in result.manifest["duplicate_messages"]:
    print(f"  {m}")
if display is not None:
    for rel in ("tables/missing_key_rows.csv", "tables/duplicates_by_keys.csv"):
        if not tables[rel].empty:
            display(tables[rel].head(20))

## Range checks (long-format output)

Unlike Stages 1A/1B (which write `flag_*_out_of_range` *columns* on
the row-level frame), Stage 4 produces:

- `range_check_summary` — one row per logical variable
  (`salinity`, `temperature`, `observed_ph`, …) with `min_allowed`,
  `max_allowed`, `n_valid`, `n_flagged`.
- `range_flags_long` — one row per (row_index, variable, direction)
  violation, with the offending value and the ID columns attached.

Long format is the right shape for analyst tooling: "show me every
salinity below 30 in 2024" is one filter; the row-level boolean
columns would need unpivoting first.

`RangePolicy` is the unified dataclass from `oa_policy.py`. Stage 4's
config supplies *wider* bounds than Stage 1A/1B (this stage asks "is
this physically possible seawater chemistry?" not "is this typical
open-ocean chemistry?").


In [ ]:
range_summary = tables["tables/range_check_summary.csv"]
print(f"Range variables checked: {len(range_summary):,}")
print(f"Flagged cells          : {counts['range_flagged_cells']:,}")
if display is not None and not range_summary.empty:
    display(range_summary)

## Strict DIC species audit

Stricter than Stage 3's diagnostic version because:

1. `abs_tol_umolkg` defaults to **5 µmol/kg** (vs Stage 3's 10) — this
   is a gate, not a screen.
2. `require_matching_units = True` — if the four species columns
   disagree on units OR fall outside `unit_equivalents`, the check is
   *not* attempted; the unit situation is flagged instead.

Five output flags:

- `flag_dic_species_audit_strict` — species sum check failed.
- `flag_dic_species_unit_mismatch_audit` — units disagree or are off list.
- `flag_dic_species_unit_missing_audit` — required units are missing.
- `flag_dic_species_audit_not_run` — required species columns are absent.
- `flag_dic_species_values_missing_audit` — species values are incomplete.


In [ ]:
print(f"DIC audit strict failures: {counts['dic_strict_fails']:,}")
dic_result = tables["tables/dic_species_audit.csv"]
if display is not None and not dic_result.empty:
    display(dic_result.head(20))

## PASS / REVIEW / FAIL classification

The substantive new logic of Stage 4. `add_readiness_status` walks
every audit-side flag, assigns reason codes, and produces the
three-tier verdict. **FAIL** beats **REVIEW** beats **PASS** wherever
multiple tiers fire.

Three reason-code columns are written:

- `analysis_audit_reason_fail` — codes that triggered FAIL (or NA).
- `analysis_audit_reason_review` — codes that triggered REVIEW (or NA).
- `analysis_audit_reason_codes` — the union of the above; this is the
  column an analyst typically filters on.


In [ ]:
print(df["analysis_audit_status"].value_counts(dropna=False).to_string())
print("\nReason-code counts:")
reason_counts = result.manifest["reason_code_counts"]
print(pd.Series(reason_counts).to_string() if reason_counts else "  (none)")

## Quick preview

In [ ]:
preview_cols = [
    c for c in [
        "record_id", "sample_id", "sample_date", "sample_month",
        "station_id", "depth_round_m",
        "salinity", "temperature_insitu_c",
        "ta_best_umolkg", "ph_best", "ph_co2sys", "dic_best_umol_kg",
        "flag_any_carbonate_issue", "flag_any_carbonate_issue_strict",
        "range_flag_count", "flag_audit_strict_dic_fail",
        "analysis_audit_status", "analysis_audit_reason_codes",
    ]
    if c in df.columns
]
if display is not None:
    display(df[preview_cols].head(20))
else:
    print(df[preview_cols].head(20).to_string(index=False))


## Write outputs

```
<OUT_DIR>/
    data/analysis_ready.csv   (and .parquet)   ◄── the final deliverable
    tables/column_inventory.csv, missingness_top40.csv, canonical_presence.csv,
           alias_resolution.csv, missing_key_rows.csv, duplicates_by_keys.csv,
           range_check_summary.csv, range_flags_long.csv, dic_species_audit.csv
    reports/report.md
    logs/manifest.json, effective_config.json
    run_bundle_manifest.json
```

`DRY_RUN = True` skips the writes. `NO_PARQUET = True` writes CSV only.
The run bundle hashes the input, the config and every file this run
wrote, so `roap verify-bundle <OUT_DIR>` can check them later.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    written = write_stage_result(result, out_root, parquet=not NO_PARQUET)
    print(f"Stage 4 complete: {len(written)} files under {out_root}")
    print(f"  -> analysis-ready CSV: {written['data/analysis_ready.csv']}")